<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/Neural-Recommendation-Personalization-Engine/blob/main/01_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')


rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# !pip install -q pandas==2.2.3
# !pip -q install -U huggingface_hub pandas pyarrow
# !pip install datasets faiss-cpu polars -q

In [5]:
!pip install -U datasets huggingface_hub

In [6]:
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/benchmark/5core/rating_only/Electronics.csv"
reviews = pd.read_csv(url)

print("Shape:", reviews.shape)
reviews.head()

Shape: (15473536, 4)


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [7]:
print('Shape:',reviews.shape,'\n\n')
print('Columns:')
print(reviews.columns.tolist(),'\n')
print('Data Types:',reviews.dtypes,'\n')
print('Missing Values:',reviews.isna().sum())
print('\nDuplicate Rows:',reviews.duplicated().sum())
print('\n Rating Distribution',reviews['rating'].value_counts().sort_index())

print("\nUnique Users:", reviews["user_id"].nunique())
print("Unique Products:", reviews["parent_asin"].nunique())

display(reviews.head())

Shape: (15473536, 4) 


Columns:
['user_id', 'parent_asin', 'rating', 'timestamp'] 

Data Types: user_id         object
parent_asin     object
rating         float64
timestamp        int64
dtype: object 

Missing Values: user_id        0
parent_asin    0
rating         0
timestamp      0
dtype: int64

Duplicate Rows: 0

 Rating Distribution rating
1.0     1287788
2.0      713558
3.0     1074820
4.0     2190347
5.0    10207023
Name: count, dtype: int64

Unique Users: 1641026
Unique Products: 368228


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [8]:
url2 = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00000-of-00010.parquet"

meta = pd.read_parquet(url2)

print("Shape:", meta.shape)
meta.head(20)

Shape: (161002, 16)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M,None,None,None
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE,None,None,None
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS,None,None,None
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW,None,None,None
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O,None,None,None
5,Sports & Outdoors,Raymarine Wi-Fish DownVision Blackbox Sonar wi...,3.5,25,[Black box Wi-Fi CHIRP DownVision sonar module...,[Transform your smartphone into a powerful CHI...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Raymarine,"[Electronics, Car & Vehicle Electronics, Marin...","{""Item Package Dimensions L x W x H"": ""9.3 x 8...",B00TX536EK,None,None,None
6,Cell Phones & Accessories,"QGHXO Band for Garmin Vivofit 4, Soft Silicone...",4.4,707,[Personalized Your Garmin Vivofit 4 Activity T...,"[Compatibility, Custom designed for your preci...",14.89,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",QGHXO,"[Electronics, Wearable Technology, Arm & Wrist...","{""Package Dimensions"": ""6.85 x 4.37 x 1.1 inch...",B07BJ7ZZL7,None,None,None
7,Industrial & Scientific,Protech iPhone 4 microscope 60X Lens with illu...,3.8,6,[iPhone 4 Microscope with White 2-LED and Dete...,[This accessory will convert your iPhone 4 int...,None,"{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",ProTech,"[Electronics, Camera & Photo, Binoculars & Sco...","{""Light Source Type"": ""LED"", ""Color"": ""Blue,Wh...",B005G99O2U,None,None,None
8,Computers,MOSISO Plastic Hard Shell Case & Keyboard Cove...,5.0,2,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",MOSISO,"[Electronics, Headphones, Earbuds & Accessorie...","{""Brand"": ""MOSISO"", ""Item Weight"": ""8 ounces"",...",B01MCZP7RF,None,None,None
9,None,"Fishfinder, Depth Finder Poly Sun Cover for 3""...",4.4,86,"[Made of poly material, Available in black, Dr...",[Affordable and convenient way to cover your s...,10.99,"{'hi_res': [None, None, None, None], 'large': ...","{'title': ['Unboxing Gamin Vivid ', 'Garmin St...",Westlake Market,"[Electronics, Car & Vehicle Electronics, Marin...","{""Item Package Dimensions L x 

In [9]:
print('Shape:',meta.shape)
print("\nColumns:",meta.columns.tolist())
print('\nData Types:',meta.dtypes)
print("\nMissing Values:",meta.isna().sum())
meta_duplicates = meta[
    [
        "main_category","title","average_rating","rating_number","price","store","details",
        "parent_asin","bought_together","subtitle","author"
    ]
].duplicated().sum()

print("Duplicate Rows:", meta_duplicates)

Shape: (161002, 16)

Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

Data Types: main_category       object
title               object
average_rating     float64
rating_number        int64
features            object
description         object
price               object
images              object
videos              object
store               object
categories          object
details             object
parent_asin         object
bought_together     object
subtitle            object
author              object
dtype: object

Missing Values: main_category        1642
title                   0
average_rating          0
rating_number           0
features                0
description             0
price                   0
images                  0
videos                  0
store                 890
categories              0


In [10]:
meta = meta.drop(columns=['bought_together','subtitle','author'])
print(meta.isna().sum())


main_category     1642
title                0
average_rating       0
rating_number        0
features             0
description          0
price                0
images               0
videos               0
store              890
categories           0
details              0
parent_asin          0
dtype: int64


In [11]:
meta['main_category'] = meta['main_category'].fillna(
    meta['main_category'].mode()[0]
)
meta['store'] = meta['store'].fillna(meta['store'].mode()[0])

In [12]:
print(meta['price'].head(20).tolist())
meta['price'] = pd.to_numeric(meta['price'],errors='coerce')
meta['price'] = meta['price'].fillna(meta['price'].median())

meta.head(20)

['None', 'None', '19.99', '9.99', '14.99', 'None', '14.89', 'None', 'None', '10.99', 'None', 'None', 'None', 'None', '8.98', 'None', '909.99', '98.49', 'None', '3.99']


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,20.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],20.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O
5,Sports & Outdoors,Raymarine Wi-Fish DownVision Blackbox Sonar wi...,3.5,25,[Black box Wi-Fi CHIRP DownVision sonar module...,[Transform your smartphone into a powerful CHI...,20.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Raymarine,"[Electronics, Car & Vehicle Electronics, Marin...","{""Item Package Dimensions L x W x H"": ""9.3 x 8...",B00TX536EK
6,Cell Phones & Accessories,"QGHXO Band for Garmin Vivofit 4, Soft Silicone...",4.4,707,[Personalized Your Garmin Vivofit 4 Activity T...,"[Compatibility, Custom designed for your preci...",14.89,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",QGHXO,"[Electronics, Wearable Technology, Arm & Wrist...","{""Package Dimensions"": ""6.85 x 4.37 x 1.1 inch...",B07BJ7ZZL7
7,Industrial & Scientific,Protech iPhone 4 microscope 60X Lens with illu...,3.8,6,[iPhone 4 Microscope with White 2-LED and Dete...,[This accessory will convert your iPhone 4 int...,20.99,"{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",ProTech,"[Electronics, Camera & Photo, Binoculars & Sco...","{""Light Source Type"": ""LED"", ""Color"": ""Blue,Wh...",B005G99O2U
8,Computers,MOSISO Plastic Hard Shell Case & Keyboard Cove...,5.0,2,[],[],20.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",MOSISO,"[Electronics, Headphones, Earbuds & Accessorie...","{""Brand"": ""MOSISO"", ""Item Weight"": ""8 ounces"",...",B01MCZP7RF
9,Computers,"Fishfinder, Depth Finder Poly Sun Cover for 3""...",4.4,86,"[Made of poly material, Available in black, Dr...",[Affordable and convenient way to cover your s...,10.99,"{'hi_res': [None, None, None, None], 'large': ...","{'title': ['Unboxing Gamin Vivid ', 'Garmin St...",Westlake Market,"[Electronics, Car & Vehicle Electronics, Marin...","{""Item Package Dimensions L x W x H"": ""7.32 x ...",B00R6R82HS


In [13]:
user_interaction = reviews.groupby('user_id').size()
item_interaction = reviews.groupby('parent_asin').size()

print("User Interaction Distribution:")
print(user_interaction.describe())

print("\nItem Interaction Distribution:")
print(item_interaction.describe())

User Interaction Distribution:
count    1.641026e+06
mean     9.429184e+00
std      8.752549e+00
min      5.000000e+00
25%      5.000000e+00
50%      7.000000e+00
75%      1.000000e+01
max      9.380000e+02
dtype: float64

Item Interaction Distribution:
count    368228.000000
mean         42.021617
std         231.315077
min           5.000000
25%           7.000000
50%          12.000000
75%          27.000000
max       44948.000000
dtype: float64


In [14]:
print("Minimum timestamp:",reviews['timestamp'].min())
print("Maximum timestamp:",reviews['timestamp'].max())

Minimum timestamp: 929311804000
Maximum timestamp: 1694509255248


In [15]:
num_users = reviews['user_id'].nunique()
num_items = reviews['parent_asin'].nunique()
num_interaction = len(reviews)

sparsity = 1 - (num_interaction / (num_users * num_items))

print("Users:", num_users)
print("Items:", num_items)
print("Interactions:", num_interaction)
print("Sparsity:", sparsity)

Users: 1641026
Items: 368228
Interactions: 15473536
Sparsity: 0.9999743930827167


In [19]:
import os

# Create NeuraRec folders in actual Google Drive
os.makedirs(
    "/content/drive/MyDrive/NeuraRec/data/raw",
    exist_ok=True
)

# Save datasets
reviews.to_parquet(
    "/content/drive/MyDrive/NeuraRec/data/raw/reviews.parquet",
    index=False
)

meta.to_parquet(
    "/content/drive/MyDrive/NeuraRec/data/raw/metadata.parquet",
    index=False
)

print("Saved!")
print(os.listdir("/content/drive/MyDrive/NeuraRec/data/raw"))

Saved!
['reviews.parquet', 'metadata.parquet']


In [ ]:
import os

print(os.path.isdir("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:20])